# 011 Memory

这是 LangGraph 学习线的第十一份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/add-memory

学习目标：

1. 区分 short-term memory 和 long-term memory
2. 学会用 checkpointer 保存同一个 `thread_id` 的短期对话状态
3. 学会用 store 保存跨 thread 的长期用户记忆
4. 理解 `thread_id` 和 `user_id` 的区别
5. 理解长对话为什么需要 trim / delete / summarize
6. 对比 LangGraph memory 和本仓库 Harness 上下文/记忆设计

## 1. 两种 Memory

官方文档把 LangGraph memory 分成两类：

| 类型 | 存在哪里 | 作用范围 | 典型用途 |
| --- | --- | --- | --- |
| short-term memory | checkpointer | 一个 `thread_id` 内 | 多轮对话上下文、当前任务状态 |
| long-term memory | store | 跨 thread、按用户或业务 namespace | 用户偏好、画像、长期事实 |

Java 类比：

```text
thread_id 像 conversationId / workflowInstanceId。
user_id 像登录用户 ID。
checkpointer 像 workflow 状态表。
store 像用户画像表或长期记忆表。
```

In [ ]:
import importlib.metadata
from dataclasses import dataclass
from operator import add
from typing import Annotated

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

## 2. Short-term memory：同一个 thread_id 内保存对话

短期记忆本质上是 graph state 被 checkpointer 保存。

下面用一个不依赖真实模型的 chat graph 演示。

`messages` 使用 reducer：

```python
Annotated[list[dict], add]
```

表示每轮输入和节点输出都会追加到 `messages`，而不是覆盖。

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[dict], add]


def short_memory_chat(state: ChatState) -> dict:
    user_messages = [
        message["content"]
        for message in state["messages"]
        if message["role"] == "user"
    ]
    answer = "我在这个 thread 里记得你说过：" + " | ".join(user_messages)
    return {"messages": [{"role": "assistant", "content": answer}]}


short_checkpointer = InMemorySaver()
short_graph = (
    StateGraph(ChatState)
    .add_node("chat", short_memory_chat)
    .add_edge(START, "chat")
    .add_edge("chat", END)
    .compile(checkpointer=short_checkpointer)
)

short_graph

## 3. 同一个 thread_id：第二轮能看到第一轮

`thread_id` 是短期记忆的边界。

In [ ]:
same_thread_config = {"configurable": {"thread_id": "short-memory-thread"}}

turn_1 = short_graph.invoke(
    {"messages": [{"role": "user", "content": "我的名字是 Bob"}]},
    same_thread_config,
)
print("turn 1 last:", turn_1["messages"][-1])

turn_2 = short_graph.invoke(
    {"messages": [{"role": "user", "content": "我刚才说了什么？"}]},
    same_thread_config,
)
print("turn 2 last:", turn_2["messages"][-1])
print("message count:", len(turn_2["messages"]))

## 4. 不同 thread_id：短期记忆隔离

换一个 `thread_id`，就像开启另一条会话。

In [ ]:
other_thread_config = {"configurable": {"thread_id": "another-short-memory-thread"}}

other_turn = short_graph.invoke(
    {"messages": [{"role": "user", "content": "这是另一条会话"}]},
    other_thread_config,
)

print("other thread last:", other_turn["messages"][-1])
print("same thread state:", short_graph.get_state(same_thread_config).values)

## 5. 管理短期记忆：trim 最近消息

短期记忆一直追加，会带来上下文窗口问题。

常见策略：

- trim：只把最近 N 条消息传给模型
- delete：从 state 中删除旧消息
- summarize：把旧消息压缩成摘要

这里先演示最简单的 trim 思想：state 里仍然保存完整历史，但传给模型前只取最近几条。

In [ ]:
def trim_messages(messages: list[dict], limit: int = 4) -> list[dict]:
    return messages[-limit:]


full_messages = short_graph.get_state(same_thread_config).values["messages"]
recent_messages = trim_messages(full_messages, limit=2)

print("full message count:", len(full_messages))
print("recent messages:", recent_messages)

## 6. Long-term memory：跨 thread 保存用户记忆

长期记忆用 `store`，不是 checkpointer。

store 适合存：

- 用户名字
- 偏好
- 长期画像
- 业务事实

关键点：长期记忆通常按 `user_id` 或业务对象 ID 做 namespace，而不是按 `thread_id`。

In [ ]:
@dataclass
class UserContext:
    user_id: str


class LongTermState(TypedDict):
    user_message: str
    answer: str


long_store = InMemoryStore()


def long_memory_node(state: LongTermState, runtime: Runtime[UserContext]) -> dict:
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "memories")
    text = state["user_message"]

    if "remember" in text.lower() or "记住" in text:
        runtime.store.put(namespace, "profile-name", {"data": "用户名字是 Bob"})

    memories = runtime.store.search(namespace)
    info = "；".join(item.value["data"] for item in memories)
    return {"answer": "长期记忆：" + info}


long_graph = (
    StateGraph(LongTermState, context_schema=UserContext)
    .add_node("long_memory", long_memory_node)
    .add_edge(START, "long_memory")
    .add_edge("long_memory", END)
    .compile(store=long_store)
)

long_graph

## 7. 同一个 user_id：跨 thread 读取长期记忆

下面第一轮让系统记住用户名字。

第二轮换一个 `thread_id`，但保持同一个 `user_id`，仍然能读到长期记忆。

In [ ]:
long_config_1 = {"configurable": {"thread_id": "long-memory-thread-1"}}
long_config_2 = {"configurable": {"thread_id": "long-memory-thread-2"}}

remember_result = long_graph.invoke(
    {"user_message": "请记住 remember：我的名字是 Bob", "answer": ""},
    long_config_1,
    context=UserContext(user_id="user-1"),
)
print("remember result:", remember_result)

recall_result = long_graph.invoke(
    {"user_message": "我叫什么名字？", "answer": ""},
    long_config_2,
    context=UserContext(user_id="user-1"),
)
print("recall result:", recall_result)

## 8. 不同 user_id：长期记忆隔离

同一个 store 里，不同 namespace 的数据互不干扰。

In [ ]:
other_user_result = long_graph.invoke(
    {"user_message": "我叫什么名字？", "answer": ""},
    {"configurable": {"thread_id": "long-memory-thread-3"}},
    context=UserContext(user_id="user-2"),
)

print("user-2 result:", other_user_result)
print("raw user-1 memories:", long_store.search(("users", "user-1", "memories")))
print("raw user-2 memories:", long_store.search(("users", "user-2", "memories")))

## 9. thread_id 和 user_id 不要混用

| ID | 用途 | 生命周期 |
| --- | --- | --- |
| `thread_id` | 一条对话或任务实例 | 短期，会话级 |
| `user_id` | 一个用户主体 | 长期，跨会话 |

如果把长期记忆按 `thread_id` 存：

```text
用户换一个会话就读不到画像。
```

如果把短期对话按 `user_id` 全部混在一起：

```text
同一个用户的多个任务会互相污染上下文。
```

## 10. 生产环境怎么落地

| 需求 | 教学实现 | 生产实现 |
| --- | --- | --- |
| 短期记忆 | `InMemorySaver` | Postgres / Redis / MongoDB / Oracle checkpointer |
| 长期记忆 | `InMemoryStore` | Postgres / Redis / MongoDB / Oracle store |
| 多进程共享 | 不支持 | 数据库或分布式存储 |
| 服务重启恢复 | 不支持 | 持久化 checkpointer / store |

官方文档里 Postgres、Redis 等实现通常需要先执行 `setup()` 初始化表结构或索引。

## 11. 和 Harness 的关系

| LangGraph | Harness 风格智能体 |
| --- | --- |
| short-term memory | 当前对话上下文、run state、ledger |
| long-term memory | 用户画像、偏好、长期事实 |
| checkpointer | approval / resume 所需的任务状态存储 |
| store | 跨会话可检索记忆 |
| trim / summarize | context budget 管理 |

关键原则：

```text
短期记忆服务当前任务。
长期记忆服务跨任务复用。
两者边界不清，就会产生上下文污染和权限风险。
```

## 12. 本讲练习

请判断下面信息应该放 short-term memory 还是 long-term memory：

1. 用户这一轮对话刚上传的文件路径。
2. 用户偏好所有回答都用中文。
3. 当前审批任务的 pending ticket。
4. 用户的姓名。
5. 当前 agent 已经执行过哪些工具。

参考答案：

1. short-term
2. long-term
3. short-term
4. long-term
5. short-term

## 13. 本讲小结

这一讲的核心：

```text
Memory = short-term thread state + long-term cross-thread store。
```

你现在应该能看懂：

- `compile(checkpointer=...)`
- `thread_id` 为什么是短期记忆边界
- `compile(store=...)`
- `Runtime[Context]` 如何读取 `user_id`
- `runtime.store.put/search` 如何保存和检索长期记忆
- 为什么长对话需要 trim / delete / summarize

下一讲可以继续学习 Subgraphs。